In [1]:
# Install the necessary libraries if you don't have them
#pip install geopy wikipedia-api requests

import json
import time
import requests
from geopy.geocoders import Nominatim
import wikipediaapi

In [2]:
input_file = "landmarks.geojson"

with open(input_file, 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)

features = geojson_data.get('features', [])
print(f"Successfully loaded {len(features)} landmarks from the file.")

Successfully loaded 306 landmarks from the file.


In [3]:
# Initialize the free geolocator
geolocator = Nominatim(user_agent="algiers_landmarks_free_scraper")

def get_name_from_coords(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), language='en', timeout=10)
        if location and 'address' in location.raw:
            address = location.raw['address']
            # Look for specific attraction names first, then fallback to street/neighborhood
            name = (address.get('historic') or address.get('tourism') or 
                    address.get('leisure') or address.get('road') or 
                    address.get('suburb'))
            
            if not name:
                name = location.address.split(',')[0]
            return name
    except Exception as e:
        print(f"Error for coordinates {lat}, {lon}: {e}")
    return None

for feature in features:
    props = feature['properties']
    # Check for English name, fallback to default name, Arabic, or French
    name = props.get('name:en') or props.get('name') or props.get('name:fr') or props.get('name:ar')
    
    if not name:
        coords = feature['geometry']['coordinates']
        lon, lat = coords[0], coords[1] 
        
        print(f"Missing name found at ({lat}, {lon}). Looking it up on OSM...")
        new_name = get_name_from_coords(lat, lon)
        
        if new_name:
            props['name'] = new_name
            print(f" -> Found: {new_name}")
        else:
            props['name'] = "Unknown Landmark"
            
        # IMPORTANT: Sleep for 1.5 seconds to respect Nominatim's strict free-usage limits (1 request/sec)
        time.sleep(1.5)
        
print("Finished filling missing names.")

Missing name found at (36.785608, 3.0610133). Looking it up on OSM...
 -> Found: نهج محمد وأحمد
Missing name found at (36.7919367, 3.0580312). Looking it up on OSM...
 -> Found: ساحة الكتاني
Missing name found at (36.7808379, 3.0504858). Looking it up on OSM...
 -> Found: Former location of the Star Fort
Missing name found at (36.766498, 3.011995). Looking it up on OSM...
 -> Found: Frais Vallon Road
Missing name found at (36.7380003, 3.0479806). Looking it up on OSM...
 -> Found: Rue Parmentier
Missing name found at (36.7743013, 3.0539874). Looking it up on OSM...
 -> Found: Boulevard Krim Belkacem
Missing name found at (36.7889775, 3.0515897). Looking it up on OSM...
 -> Found: Rue Hassena Ahmed شارع أحمد حسينة
Missing name found at (36.7844021, 3.0640788). Looking it up on OSM...
 -> Found: شارع أنكور
Missing name found at (36.7736201, 3.056456). Looking it up on OSM...
 -> Found: Boulevard Yakoub Saïd
Missing name found at (36.7747168, 3.0553013). Looking it up on OSM...
 -> Found:

In [4]:
# Initialize Wikipedia API with a proper user agent
wiki_wiki = wikipediaapi.Wikipedia('AlgiersLandmarkProject (ayoub42psg@gmail.com)', 'en')

def get_wikipedia_info(landmark_name):
    info = {"description": None, "image_url": None}
    if not landmark_name or landmark_name == "Unknown Landmark":
        return info
        
    try:
        page = wiki_wiki.page(landmark_name)
        if not page.exists():
            # Try adding "Algiers" to specify the location
            page = wiki_wiki.page(f"{landmark_name} Algiers")
            
        if page.exists():
            # Get the first paragraph
            info["description"] = page.summary.split('\n')[0]
            
            # Fetch image URL using the free Wikimedia API
            wiki_api_url = f"https://en.wikipedia.org/w/api.php?action=query&titles={page.title}&prop=pageimages&format=json&pithumbsize=800"
            response = requests.get(wiki_api_url).json()
            pages = response.get('query', {}).get('pages', {})
            
            for page_id, page_data in pages.items():
                if 'thumbnail' in page_data:
                    info["image_url"] = page_data['thumbnail']['source']
                    break
    except Exception as e:
        pass # Ignore errors to keep script running smoothly
        
    return info

for feature in features:
    props = feature['properties']
    name = props.get('name:en') or props.get('name')
    
    if name and name != "Unknown Landmark":
        print(f"Fetching Wiki data for: {name}...")
        wiki_data = get_wikipedia_info(name)
        
        props['description'] = wiki_data['description']
        props['image_url'] = wiki_data['image_url']

print("Finished fetching Wikipedia descriptions and images.")

Fetching Wiki data for: City of sciences Algiers...
Fetching Wiki data for: Musée Public National de l’Enluminure de la Miniature et de la Calligraphie المتحف العمومي الوطني للزخرفة والمنمنمات وفن الخط...
Fetching Wiki data for: نهج محمد وأحمد...
Fetching Wiki data for: Dar El Salam...
Fetching Wiki data for: Prague Garden...
Fetching Wiki data for: ساحة الكتاني...
Fetching Wiki data for: El Hamma Experimental Garden...
Fetching Wiki data for: حديقة تفاريتي...
Fetching Wiki data for: Jardin de Beyrouth...
Fetching Wiki data for: Jardin de Tunis...
Fetching Wiki data for: Port-Saïd Square...
Fetching Wiki data for: Former location of the Star Fort...
Fetching Wiki data for: Frais Vallon Road...
Fetching Wiki data for: Jardin Olof Palme...
Fetching Wiki data for: Jardin Ibn Badis...
Fetching Wiki data for: Said Touati...
Fetching Wiki data for: Rue Parmentier...
Fetching Wiki data for: Jardin Sofia...
Fetching Wiki data for: Domaine Tamzali...
Fetching Wiki data for: Jardin Lavigerie...


In [5]:
def unify_category(properties):
    """Creates a unified category based on existing OpenStreetMap tags"""
    tourism = properties.get('tourism', '')
    historic = properties.get('historic', '')
    leisure = properties.get('leisure', '')
    
    if 'museum' in tourism:
        return 'Museum'
    elif 'garden' in leisure or 'park' in leisure:
        return 'Park & Garden'
    elif 'monument' in historic or 'ruins' in historic or 'archaeological_site' in historic:
        return 'Historic Monument'
    elif 'artwork' in tourism:
        return 'Artwork & Statue'
    elif 'viewpoint' in tourism:
        return 'Viewpoint'
    else:
        return 'Other Attraction'

def get_free_opening_hours(properties):
    """Extracts opening hours if OpenStreetMap contributors have added them"""
    # OSM uses the 'opening_hours' tag standardly
    if 'opening_hours' in properties:
        return properties['opening_hours']
    return "Hours not available on OpenStreetMap"

for feature in features:
    props = feature['properties']
    
    # 1. Unify Category based on free OSM data
    props['unified_category'] = unify_category(props)
    
    # 2. Extract opening hours if they exist in the raw Overpass data
    props['opening_hours'] = get_free_opening_hours(props)

print("Finished unifying categories and extracting free opening hours.")

Finished unifying categories and extracting free opening hours.


In [6]:
output_file = "enriched_landmarks_free.geojson"

# Update the main dictionary with the updated features
geojson_data['features'] = features

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(geojson_data, f, ensure_ascii=False, indent=4)

print(f"Success! 100% Free enriched data has been saved to {output_file}")

Success! 100% Free enriched data has been saved to enriched_landmarks_free.geojson
